# Centralized Model Manager & Zoo Playground
This notebook serves as the single source of truth for downloading, converting, verifying, and deploying embedding models for the 3D Furniture Multimodal Search Engine.

### Production Constraints Enforced:
1. Commercial Compliance: Only Apache 2.0 or MIT licensed models are permitted.

2. Generic Decoupling: Files are auto-renamed to generic names (clip_visual.onnx, clip_text.onnx) so backend code requires zero modification when models swap.

3. Monorepo Structure: All assets are compiled directly into the root models/ directory shared across Python and C++.

# (Model Playground MVP)
Ten notebook służy do automatycznego pobierania, weryfikacji i przełączania modeli CLIP dla wielomodalnej wyszukiwarki mebli 3D.

Dzięki niemu możemy elastycznie testować różne architektury (mniejsze/szybsze lub większe/dokładniejsze), zachowując spójne nazewnictwo plików.

In [1]:
# 1. Install required framework dependencies
!pip install huggingface_hub onnx optimum[onnxruntime] tqdm --quiet


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import sys

!{sys.executable} -m pip install onnx


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
import sys
import json
import shutil
import onnx
from huggingface_hub import hf_hub_download, snapshot_download

# Resolve standard Monorepo paths (Targeting clip_mvp_app/models)
PROJECT_ROOT = os.path.dirname(os.getcwd())
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")
TOKENIZER_DIR = os.path.join(MODELS_DIR, "clip_tokenizer")

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(TOKENIZER_DIR, exist_ok=True)

print(f"✅ Environment initialized.")
print(f"📁 Target deployment directory: {MODELS_DIR}")

✅ Environment initialized.
📁 Target deployment directory: c:\Users\CAD\Desktop\MyApp\models


# Lista dostępnych modeli i ich konfiguracji
W tej sekcji definiujemy mapę modeli dostępnych na Hugging Face, które pasują do naszego zastosowania. Możesz tu łatwo dopisywać nowe pozycje w przyszłości.

- Option 1 (Baseline Multilingual): Sprawdzony, stabilny model radzący sobie z językiem polskim.

- Option 2 (Pure English / Fast Test): Klasyczny model od OpenAI, idealny do szybkich testów wydajnościowych (mniejszy rozmiar).

## 🔍 Przewodnik Dewelopera: Jak rozbudowywać Model Zoo?

Ten notebook został zaprojektowany tak, aby umożliwić bezproblemową wymianę modeli w przyszłości. Jeśli chcesz przetestować nową architekturę (np. mniejszą/szybszą lub bardziej precyzyjną), musisz odpowiednio zmapować ją w słowniku `MODEL_ZOO`.

### 1. Jak szukać kompatybilnych modeli ONNX?
Najbezpieczniejszym i najszybszym źródłem gotowych modeli w formacie ONNX jest profil **`Xenova`** na Hugging Face (odpowiedzialny za oficjalny ekosystem *Transformers.js*).
1. Wejdź na [huggingface.co/Xenova](https://huggingface.co/Xenova).
2. Wyszukaj interesującą Cię architekturę (np. wpisując `clip` lub `siglip`).
3. Wejdź w dane repozytorium i skopiuj jego identyfikator (np. `Xenova/clip-vit-base-patch32`). To będzie Twoje `repo_id`.

### 2. Sprawdzanie struktury plików grafu
Przed dodaniem wpisu do kodu przejdź do zakładki **"Files and versions"** wybranego modelu:
* Wejdź do folderu `onnx/`.
* Upewnij się, że znajdują się tam pliki dla enkodera wizualnego oraz tekstowego.
* Podaj ścieżki do nich w słowniku (zazwyczaj jest to `"onnx/vision_model.onnx"` oraz `"onnx/text_model.onnx"`).

---

## ⚠️ Bardzo Ważne: Audyt Licencji Komercyjnej (Compliance Checklist)

Ponieważ nasza aplikacja jest rozwijana jako **produkt komercyjny**, każdy nowy model dodawany do tego projektu **MUSI** pozwalać na swobodne wykorzystanie biznesowe bez ryzyka prawnego. Przed pobraniem jakiegokolwiek modelu bezwzględnie sprawdź tag `License` na karcie modelu (Model Card).

### Ściągawka Licencyjna dla Zespołu:

| Status komercyjny | Typ Licencji | Czy możemy użyć w kodzie? | Opis |
| :--- | :--- | :--- | :--- |
| 🟢 **Zgoda (100% bezpieczne)** | **MIT / Apache 2.0** | **TAK** | Licencje wysoce liberalne. Możemy modyfikować model, zamykać nasz kod i sprzedawać aplikację klientom bez opłat licencyjnych. |
| 🟡 **Zgoda warunkowa** | **OpenRAIL / CC-BY-4.0** | **TAK (Po weryfikacji)** | Można użyć komercyjnie, ale wymaga to dopisania autora do dokumentacji aplikacji (`CC-BY`) lub przestrzegania zasad etycznego użytku (`OpenRAIL`). |
| 🔴 **BEZWZGLĘDNY ZAKAZ** | **CC-BY-NC (dowolna wersja)** | 🛑 **NIE!** | Końcówka **"-NC"** oznacza *Non-Commercial* (Użycie Niekomercyjne). Wdrożenie takiego modelu w komercyjnym oprogramowaniu grozi poważnymi konsekwencjami prawnymi i finansowymi. |

> 💡 **Złota zasada:** Nasz obecny baseline (`Xenova/clip-ViT-B-32-multilingual-v1`) bazuje na architekturach OpenAI oraz Meta i jest wydany na licencjach MIT oraz Apache 2.0, co czyni go w 100% bezpiecznym dla celów biznesowych.

In [14]:
MODEL_ZOO = {
    "siglip2_base_multilingual": {
        "repo_id": "google/siglip2-base-patch16-256",
        "method": "optimum_export",
        "task": "feature-extraction",  # <--- Jawna informacja dla eksportera
        "vector_dimensions": 768,
        "required_resolution": "256x256",
        "description": "Google SigLIP 2 Base (Apache 2.0) - High precision Polish/English alignment."
    },
    "siglip2_large_multilingual": {
        "repo_id": "google/siglip2-large-patch16-256",
        "method": "optimum_export",
        "task": "feature-extraction",
        "vector_dimensions": 1024,
        "required_resolution": "256x256",
        "description": "Google SigLIP 2 Large (Apache 2.0) - Higher quality multilingual retrieval."
    },
    "multilingual_vit_b32_baseline": {
        "repo_id": "Xenova/clip-ViT-B-32-multilingual-v1",
        "method": "xenova_download",
        "task": "automatic",
        "vector_dimensions": 512,
        "required_resolution": "224x224",
        "description": "M-CLIP Base (MIT) - Legacy multi-lingual baseline."
    }
}



# "KLUCZ_DLA_CIEBIE": {
#     "repo_id": "UŻYTKOWNIK/NAZWA_REPO_Z_URL",
#     "vision_file": "SZYBKA_SCIEZKA_Z_TABU_FILES/vision.onnx",
#     "text_file": "SZYBKA_SCIEZKA_Z_TABU_FILES/text.onnx",
#     "description": "Twoja własna notatka, żebyś pamiętał co to za model"
# }

SELECTED_MODEL_KEY = "siglip2_large_multilingual"
active_config = MODEL_ZOO[SELECTED_MODEL_KEY]
print(f"🎯 Target Model Selected: {active_config['description']}")

🎯 Target Model Selected: Google SigLIP 2 Large (Apache 2.0) - Higher quality multilingual retrieval.


# ⚙️ Execution Pipeline (Download vs. Export)
- This block dynamically executes the correct acquisition strategy based on the model's design, abstracts away vendor folder structures, and extracts the raw assets.

- Ten proces pobiera wybrane pliki z Hugging Face, automatycznie zmienia ich nazwy na format generyczny oczekiwany przez konfigurację backendu (clip_visual_vitb32.onnx, clip_text_vitb32.onnx) oraz generuje aktualny plik model_manifest.json.

In [16]:
import shutil
print(shutil.which("optimum-cli"))

c:\Users\CAD\Desktop\sciezki_manager\venv\Scripts\optimum-cli.EXE


In [21]:
import subprocess
import sys
import os
import shutil

FINAL_UNIFIED_PATH = os.path.join(MODELS_DIR, "model.onnx")
FINAL_VISUAL_PATH = os.path.join(MODELS_DIR, "clip_visual.onnx")
FINAL_TEXT_PATH = os.path.join(MODELS_DIR, "clip_text.onnx")

# Clean up any old files from previous legacy tests
for path in [FINAL_UNIFIED_PATH, FINAL_VISUAL_PATH, FINAL_TEXT_PATH]:
    if os.path.exists(path):
        os.remove(path)

if active_config["method"] == "xenova_download":
    print("📥 Fetching pre-converted vendor ONNX graphs...")
    vis_temp = hf_hub_download(repo_id=active_config["repo_id"], filename="onnx/vision_model.onnx", local_dir=MODELS_DIR)
    txt_temp = hf_hub_download(repo_id=active_config["repo_id"], filename="onnx/text_model.onnx", local_dir=MODELS_DIR)
    shutil.move(os.path.join(MODELS_DIR, "onnx", "vision_model.onnx"), FINAL_VISUAL_PATH)
    shutil.move(os.path.join(MODELS_DIR, "onnx", "text_model.onnx"), FINAL_TEXT_PATH)
    shutil.rmtree(os.path.join(MODELS_DIR, "onnx"), ignore_errors=True)
    
    print("📥 Syncing tokenizer configurations...")
    snapshot_download(repo_id=active_config["repo_id"], allow_patterns=["*.json", "*.txt"], ignore_patterns=["onnx/*"], local_dir=TOKENIZER_DIR)

elif active_config["method"] == "optimum_export":
    print("⚡ Executing local PyTorch to ONNX compilation via Hugging Face Optimum...")
    TEMP_EXPORT_DIR = os.path.join(MODELS_DIR, "temp_export")
    
    venv_bin_dir = os.path.dirname(sys.executable)
    # optimum_cli_exe = os.path.join(venv_bin_dir, "optimum-cli.exe")
    optimum_cli_exe = shutil.which("optimum-cli")  # More robust way to find the executable
    if not os.path.exists(optimum_cli_exe):
        optimum_cli_exe = os.path.join(venv_bin_dir, "optimum-cli")

    cmd = [
        optimum_cli_exe, "export", "onnx",
        "--model", active_config["repo_id"],
        "--task", active_config["task"],
        TEMP_EXPORT_DIR
    ]
    
    process = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    
    if process.returncode != 0:
        print("\n🛑 --- TERMINAL ERROR LOG --- 🛑")
        print(process.stdout)
        print(process.stderr)
        print("---------------------------------")
        raise RuntimeError("❌ ONNX Compilation failed.")
        
    generated_files = os.listdir(TEMP_EXPORT_DIR)
    source_unified = os.path.join(TEMP_EXPORT_DIR, "model.onnx")
    
print("\n=== Export directory contents ===")

for f in os.listdir(TEMP_EXPORT_DIR):
    full_path = os.path.join(TEMP_EXPORT_DIR, f)

    if os.path.isfile(full_path):
        size_mb = os.path.getsize(full_path) / (1024 * 1024)
        print(f"FILE: {f} ({size_mb:.2f} MB)")
    else:
        print(f"DIR : {f}")

print("===============================\n")

source_unified = os.path.join(TEMP_EXPORT_DIR, "model.onnx")

if not os.path.exists(source_unified):
    raise FileNotFoundError(
        f"❌ Expected model.onnx was not found.\n"
        f"Directory contains: {os.listdir(TEMP_EXPORT_DIR)}"
    )

# Copy main ONNX
shutil.copy2(source_unified, FINAL_UNIFIED_PATH)

# Copy external weights if present
source_data = os.path.join(TEMP_EXPORT_DIR, "model.onnx_data")

if os.path.exists(source_data):
    final_data = os.path.join(MODELS_DIR, "model.onnx_data")

    shutil.copy2(source_data, final_data)

    print("✅ model.onnx_data copied.")
else:
    print("ℹ️ No model.onnx_data generated.")

print("✅ Unified model exported successfully.")
    
# Migrate tokenizer configurations
for item in os.listdir(TEMP_EXPORT_DIR):
    item_path = os.path.join(TEMP_EXPORT_DIR, item)
    if not item.endswith(".onnx"):
        dest_path = os.path.join(TOKENIZER_DIR, item)
        if os.path.isdir(item_path):
            shutil.copytree(item_path, dest_path, dirs_exist_ok=True)
        else:
            shutil.copy2(item_path, dest_path)
                
print(f"\n📂 Export left for inspection:")
print(TEMP_EXPORT_DIR)

print("\n🎉 Pipeline complete. Maximum space optimization achieved!")

⚡ Executing local PyTorch to ONNX compilation via Hugging Face Optimum...

=== Export directory contents ===
FILE: chat_template.jinja (0.00 MB)
FILE: config.json (0.00 MB)
FILE: model.onnx (0.94 MB)
FILE: model.onnx_data (3362.76 MB)
FILE: special_tokens_map.json (0.00 MB)
FILE: tokenizer.json (32.77 MB)
FILE: tokenizer_config.json (0.05 MB)

✅ model.onnx_data copied.
✅ Unified model exported successfully.

📂 Export left for inspection:
c:\Users\CAD\Desktop\MyApp\models\temp_export

🎉 Pipeline complete. Maximum space optimization achieved!


# 📝 Runtime Manifest Verification
Generates a frozen model_manifest.json descriptor file within the deployment folder so runtime client components can audit the active database parameters dynamically.

In [18]:
manifest = {
    "deployment_identity": SELECTED_MODEL_KEY,
    "original_hub_repo": active_config["repo_id"],
    "is_unified_model": (active_config["method"] == "optimum_export"),
    "expected_tensor_shapes": {
        "embedding_dimensions": active_config["vector_dimensions"],
        "preprocessing_image_resolution": active_config["required_resolution"]
    },
    "assets": {
        "model_file": "model.onnx" if active_config["method"] == "optimum_export" else None,
        "visual_graph": None if active_config["method"] == "optimum_export" else "clip_visual.onnx",
        "text_graph": None if active_config["method"] == "optimum_export" else "clip_text.onnx",
        "tokenizer_config": "clip_tokenizer/"
    }
}

manifest_path = os.path.join(MODELS_DIR, "model_manifest.json")
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

print(f"💾 Production manifest optimized and saved to: {manifest_path}")

💾 Production manifest optimized and saved to: c:\Users\CAD\Desktop\MyApp\models\model_manifest.json
